# 0. Document Loaders Overview (BaseLoader + Document)

Before we meet individual loaders, let's learn the **two things every loader has in common**. Learn
these once and every loader afterward becomes easy.

1. **`Document`** — the standard "box" that loaded data is put into.
2. **`BaseLoader`** — the parent class every loader inherits from (gives them `.load()` /
   `.lazy_load()`).

---

## 1. Simple Definition

> **Kid version:** Imagine you collect toys from many different boxes — a red box, a blue box, a
> shoebox. To keep your room tidy, you put **every toy into the same kind of clear plastic bin** with
> a **label** on it. Now, no matter where a toy came from, you handle all bins the same way.
>
> - The **clear plastic bin** = a `Document`.
> - The **toy inside** = the actual text (`page_content`).
> - The **label** = the `metadata` (where it came from, page number, row number…).
> - The **person who fills the bins** = a **loader** (`BaseLoader`).

**Professional definition:** A document loader reads a data source and returns a **list of
`Document` objects**. Each `Document` holds the extracted text plus metadata. Every loader is a
subclass of `BaseLoader`, so they all share the same `.load()` interface.

---

## 2. Why Does It Exist?

**The problem:** Your data lives in many formats — `.txt`, `.csv`, `.json`, `.pdf`, web pages,
Notion, Slack, databases… An LLM can't read a PDF's binary bytes, and the rest of LangChain
(splitters, vector stores) can't work with 50 different custom shapes.

### Before LangChain (do it by hand, differently for each format)

```python
# text
text = open("notes.txt").read()

# csv
import csv
rows = list(csv.DictReader(open("data.csv")))

# pdf
import pypdf
pdf = pypdf.PdfReader("report.pdf")
pages = [p.extract_text() for p in pdf.pages]

# web
import requests, bs4
html = requests.get(url).text
text = bs4.BeautifulSoup(html, "html.parser").get_text()
```

Every source needs different code, different libraries, and produces a **different shape**. Nothing
downstream can be reused.

### After LangChain

```python
from langchain_community.document_loaders import TextLoader, CSVLoader, PyPDFLoader, WebBaseLoader

docs1 = TextLoader("notes.txt").load()
docs2 = CSVLoader("data.csv").load()
docs3 = PyPDFLoader("report.pdf").load()
docs4 = WebBaseLoader(url).load()
# ALL of these return the SAME thing: list[Document]
```

One method (`.load()`), one output shape (`list[Document]`), for **every** source. Now splitting,
embedding, and storing are identical no matter where the data came from.

---

## 3. Real-Life Analogy

An **airport translator** 🛬. Travelers arrive speaking many languages (txt, csv, pdf, HTML). The
translator converts everyone into **one common language** (`Document`) so the immigration officers
(the rest of your pipeline) only ever deal with one language.

Or a **shipping warehouse**: goods arrive in all kinds of packaging, but the warehouse repacks
everything into **identical labeled boxes** so forklifts, shelves, and trucks can handle them
uniformly.

---

## 4. Where It Fits in LangChain Architecture

```
        BaseLoader                      ← the ancestor of ALL loaders
            │                             defines: load(), lazy_load(), aload(), load_and_split()
   ┌────────┼───────────┬───────────┬───────────┐
   ▼        ▼           ▼           ▼           ▼
TextLoader CSVLoader JSONLoader  PyPDFLoader  WebBaseLoader   ... (100s more)
```

Every loader is a `BaseLoader`. So once you know how *one* loader is used, you know how they **all**
are used — only the constructor arguments differ.

And each loader outputs `Document` objects:

```
Document
 ├── page_content : str   ← the actual text
 └── metadata     : dict  ← info about the text (source, page, row, ...)
```

---

## 5. Internal Working

```
  SOURCE (file / url)
        │
        ▼
  ┌──────────────────────┐
  │  Loader              │  1. OPEN the source (file handle / HTTP request)
  │  (a BaseLoader)      │  2. PARSE the raw bytes into text (format-specific)
  │                      │  3. WRAP text + info into Document object(s)
  └──────────────────────┘
        │
        ▼
  list[Document]
     [ Document(page_content="...", metadata={"source": "..."}),
       Document(page_content="...", metadata={...}), ... ]
```

The only part that changes between loaders is **step 2 (parse)**. Opening and wrapping are the same
idea everywhere.

---

## 6. The Core Pieces (shared by every loader)

### The `Document` object

**Definition:** The standard container for a piece of loaded content.

**Why it exists:** So everything downstream (splitters, embeddings, vector stores) has *one* shape to
understand.

**Real-life use case:** The labeled plastic bin every toy goes into.

In [1]:
from langchain_core.documents import Document

doc = Document(page_content="LangChain makes LLM apps easier.",
               metadata={"source": "notes.txt", "author": "Sam"})

print(doc.page_content)   # the text
print(doc.metadata)       # {"source": "notes.txt", "author": "Sam"}

LangChain makes LLM apps easier.
{'source': 'notes.txt', 'author': 'Sam'}


- **`page_content`** *(str)* — the actual text the LLM will eventually read.
- **`metadata`** *(dict)* — data *about* the text: `source`, `page`, `row`, timestamps, etc.
  Metadata is gold for **filtering** ("only search page 3 of this PDF") and **citations**
  ("this answer came from `report.pdf`, page 5").

### `.load()`

**Definition:** Reads the **entire** source and returns `list[Document]` all at once.

**Why it exists:** The simplest, most common way to get your data in.

**When developers use it:** Small/medium sources that fit comfortably in memory.

In [4]:
from langchain_community.document_loaders import CSVLoader

docs = CSVLoader(r"knowledge-source\organizations.csv").load()   
print(len(docs), type(docs[0]))         

1000 <class 'langchain_core.documents.base.Document'>


### `.lazy_load()`

**Definition:** Returns a **generator** that yields one `Document` at a time, instead of all at once.

**Why it exists:** Huge files (a 5,000-page PDF, a giant CSV) can blow up memory if loaded fully.
Lazy loading streams them piece by piece.

**When developers use it:** Big data, streaming pipelines, low-memory environments.

**Real-life use case:** Reading a giant book **one page at a time** instead of photocopying all 5,000
pages before you start.

In [3]:
from langchain_community.document_loaders import CSVLoader

docs = CSVLoader(r"knowledge-source\organizations.csv").lazy_load()   
docs  

<generator object CSVLoader.lazy_load at 0x000001DAEA66C9D0>

### `.aload()` / `.alazy_load()`

**Definition:** **Async** versions of the above, for async apps (FastAPI, high concurrency).